In [ ]:
import pandas as pd
import numpy as np
import string
import gc
import io
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from collatex import Collation, collate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel
import torch
import warnings
warnings.filterwarnings('ignore')


nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('omw-1.4', quiet=True)
lemmatizer = WordNetLemmatizer()



def get_tfidf_features(texts):
    """Baseline: TF-IDF Character n-grams"""
    vec = TfidfVectorizer(analyzer='char', ngram_range=(2,4), max_features=3000, sublinear_tf=True)
    return vec.fit_transform(texts).toarray()

def get_readability_features(texts):
    """Updated Readability Features"""
    features = []
    for text in texts:
        words = word_tokenize(text)
        alpha_words = [w for w in words if w.isalpha()]
        sentences = sent_tokenize(text)
        
        sent_len = len(alpha_words) / max(len(sentences), 1)
        avg_word_len = sum(len(w) for w in alpha_words) / max(len(alpha_words), 1)
        ttr = len(set(w.lower() for w in alpha_words)) / max(len(alpha_words), 1)
        char_len = len(text)
        punct_count = sum(1 for char in text if char in string.punctuation)
        punct_ratio = punct_count / max(len(text), 1)
        
        features.append([sent_len, avg_word_len, ttr, char_len, punct_ratio])
    return np.array(features)

def get_enhanced_features(texts):
    """Enhanced Morphological Features"""
    features = []
    for text in texts:
        words = word_tokenize(text)
        alpha_words = [w for w in words if w.isalpha()]
        tags = pos_tag(alpha_words)
        
        lemmas = [lemmatizer.lemmatize(w.lower()) for w in alpha_words]
        lemma_ratio = len(set(lemmas)) / max(len(lemmas), 1)
        
        pos_counts = {'NN': 0, 'VB': 0, 'JJ': 0, 'RB': 0}
        for _, tag in tags:
            if tag.startswith('NN'): pos_counts['NN'] += 1
            elif tag.startswith('VB'): pos_counts['VB'] += 1
            elif tag.startswith('JJ'): pos_counts['JJ'] += 1
            elif tag.startswith('RB'): pos_counts['RB'] += 1
            
        total_pos = sum(pos_counts.values()) + 1e-5
        pos_ratios = [pos_counts[k] / total_pos for k in ['NN', 'VB', 'JJ', 'RB']]
        
        features.append([lemma_ratio] + pos_ratios)
    return np.array(features)

def get_distilbert_features(texts):
    """DistilBERT [CLS] Embeddings"""
    print("    Loading DistilBERT...")
    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()

    embeddings = []
    
    
    texts_list = [str(t) if str(t).strip().lower() != 'nan' else "empty text" for t in texts]
    
    for i in range(0, len(texts_list), 32):
        batch = texts_list[i:i+32]
        inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=64).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
            
    return np.vstack(embeddings).astype(np.float32)

def get_variant_features(df):
    """CollateX Variant Features per STORY"""
    print("    Running CollateX alignment...")
    story_feats = []
    
    for story_id, group in df.groupby('STORY'):
        if len(group) < 2: continue
        
        collation = Collation()
        titles = []
        for i, (_, row) in enumerate(group.iterrows()):
            t = str(row['TITLE']).strip()
            if len(t) < 3: continue
            collation.add_plain_witness(f"w{i}", t)
            titles.append(t)
            
        if len(titles) < 2: continue
        
        try:
            csv_str = collate(collation, output='csv')
            align = pd.read_csv(io.StringIO(csv_str), index_col=0)
            
            counts = {'same':0, 'replace':0, 'insert':0, 'delete':0}
            lengths = [len(t.split()) for t in titles]
            
            for col in align.columns:
                tokens = [str(v).strip() for v in align[col] if pd.notna(v) and str(v).strip()!='']
                if len(tokens) == 0: continue
                if len(tokens) == 1: 
                    counts['insert'] += 1 
                else:
                    unique = list(set(tokens))
                    if len(unique) == 1: counts['same'] += 1
                    else: counts['replace'] += 1
                    
            total = sum(counts.values()) + 1e-5
            max_len = max(lengths) + 1e-5
            min_len = min(lengths) + 1e-5
            
            story_feats.append({
                'STORY': story_id,
                'same_ratio': counts['same'] / total,
                'replace_ratio': counts['replace'] / total,
                'insert_ratio': counts['insert'] / total,
                'delete_ratio': counts['delete'] / total, 
                'compression_ratio': min_len / max_len,
                'length_diff_ratio': abs(lengths[0] - lengths[1]) / max_len
            })
        except: pass
        
    return pd.DataFrame(story_feats) if story_feats else pd.DataFrame(columns=['STORY','same_ratio','replace_ratio','insert_ratio','delete_ratio','compression_ratio','length_diff_ratio'])


def run_ablation_study(filepath, dataset_name):
    print(f"\n{'='*70}")
    print(f" ABLATION STUDY: {dataset_name.upper()}")
    print(f"{'='*70}")
    
    df = pd.read_csv(filepath)
    df = df.dropna(subset=['TITLE'])
    df['TITLE'] = df['TITLE'].astype(str)
    texts = df['TITLE'].values
    labels = df['PUBLISHER'].values
    
    print(" Extracting Baseline (TF-IDF Char)...")
    X_tfidf = get_tfidf_features(texts)
    
    print(" Extracting Readability Features...")
    X_read = get_readability_features(texts)
    
    print(" Extracting Enhanced Features...")
    X_enh = get_enhanced_features(texts)
    
    print(" Extracting Variant Features (CollateX)...")
    variant_df = get_variant_features(df)
    df = df.merge(variant_df, on='STORY', how='left').fillna(0)
    X_var = df[['same_ratio', 'replace_ratio', 'insert_ratio', 'delete_ratio', 'compression_ratio', 'length_diff_ratio']].values
    
    print(" Extracting DistilBERT Features...")
    X_distil = get_distilbert_features(texts)
    
    steps = {
        '1. Baseline (TF-IDF Char)': X_tfidf,
        '2. Baseline + Readability': np.hstack([X_tfidf, X_read]),
        '3. Baseline + Readability + Variant': np.hstack([X_tfidf, X_read, X_var]),
        '4. Full Ablation (All Features)': np.hstack([X_tfidf, X_read, X_var, X_enh]),
        '5. Ultimate Hybrid (Full + DistilBERT)': np.hstack([X_tfidf, X_read, X_var, X_enh, X_distil])
    }
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        'Linear SVM': LinearSVC(random_state=42, max_iter=2000, class_weight='balanced')
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    results = []
    
    print("\n Running Ablation Steps...\n")
    for step_name, X_step in steps.items():
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_step)
        
        for model_name, model in models.items():
            acc = cross_val_score(model, X_scaled, labels, cv=cv, scoring='accuracy').mean()
            f1 = cross_val_score(model, X_scaled, labels, cv=cv, scoring='f1_macro').mean()
            results.append({'Step': step_name, 'Model': model_name, 'Accuracy': acc, 'F1_Macro': f1})
            print(f"   {step_name:<40} | {model_name:<20} | Acc: {acc:.4f} | F1: {f1:.4f}")
            
    res_df = pd.DataFrame(results)
    
    print(f"\n ABLATION SUMMARY ({dataset_name.upper()}):")
    summary = res_df.groupby('Step')[['Accuracy', 'F1_Macro']].max().reset_index()
    print(summary.to_string(index=False))
    
    del X_tfidf, X_read, X_var, X_enh, X_distil
    gc.collect()
    return res_df


duo_res = run_ablation_study('clean_duo_data.csv', 'DUO')
trio_res = run_ablation_study('clean_trio_data.csv', 'TRIO')

print("\n" + "="*70)
print(" FINAL COMPARISON: BEST F1 PER STEP")
print("="*70)
final_comp = pd.concat([
    duo_res.groupby('Step')['F1_Macro'].max().reset_index().rename(columns={'F1_Macro': 'DUO_F1'}),
    trio_res.groupby('Step')['F1_Macro'].max().reset_index().rename(columns={'F1_Macro': 'TRIO_F1'})
], axis=1)
print(final_comp[['Step', 'DUO_F1', 'TRIO_F1']].to_string(index=False))


 ABLATION STUDY: DUO
📝 Extracting Baseline (TF-IDF Char)...
📊 Extracting Readability Features...
🧠 Extracting Enhanced Features...
🔗 Extracting Variant Features (CollateX)...
   ⚙️ Running CollateX alignment...
 Extracting DistilBERT Features...
    Loading DistilBERT...

 Running Ablation Steps...

  ✅ 1. Baseline (TF-IDF Char)                | Logistic Regression  | Acc: 0.8588 | F1: 0.8489
  ✅ 1. Baseline (TF-IDF Char)                | Linear SVM           | Acc: 0.8422 | F1: 0.8326
  ✅ 2. Baseline + Readability                | Logistic Regression  | Acc: 0.8666 | F1: 0.8570
  ✅ 2. Baseline + Readability                | Linear SVM           | Acc: 0.8510 | F1: 0.8418
  ✅ 3. Baseline + Readability + Variant      | Logistic Regression  | Acc: 0.8685 | F1: 0.8590
  ✅ 3. Baseline + Readability + Variant      | Linear SVM           | Acc: 0.8454 | F1: 0.8363
  ✅ 4. Full Ablation (All Features)          | Logistic Regression  | Acc: 0.8692 | F1: 0.8595
  ✅ 4. Full Ablation (All Feature